# China V1 容量估算复核

本 notebook 只使用 Python 标准库，复核报告中的全国像元数、embedding 和训练样本缓存体量。体量使用十进制 TB。

In [1]:
china_km2 = 9_600_000
pixels_2m = china_km2 * 1_000_000 / (2 * 2)
pixels_10m = china_km2 * 1_000_000 / (10 * 10)
patches_1280m = china_km2 / (1.28 * 1.28)
print(f'全国 2m 像元: {pixels_2m:,.0f}')
print(f'全国 10m 像元: {pixels_10m:,.0f}')
print(f'1280m patch 数: {patches_1280m:,.0f}')

全国 2m 像元: 2,400,000,000,000
全国 10m 像元: 96,000,000,000
1280m patch 数: 5,859,375


In [2]:
def tb(values, bytes_per_value=1):
    return values * bytes_per_value / 1e12

print(f'2m 4-band uint16 raw: {tb(pixels_2m * 4, 2):.3f} TB')
print(f'2m 64D int8 one snapshot: {tb(pixels_2m * 64):.3f} TB')
print(f'2m 64D fp16 one snapshot: {tb(pixels_2m * 64, 2):.3f} TB')
print(f'2m 64D int8 eight quarters: {tb(pixels_2m * 64) * 8 / 1000:.3f} PB')
print(f'10m 64D int8 one snapshot: {tb(pixels_10m * 64):.3f} TB')
print(f'10m 64D int8 eight quarters: {tb(pixels_10m * 64) * 8:.3f} TB')

2m 4-band uint16 raw: 19.200 TB
2m 64D int8 one snapshot: 153.600 TB
2m 64D fp16 one snapshot: 307.200 TB
2m 64D int8 eight quarters: 1.229 PB
10m 64D int8 one snapshot: 6.144 TB
10m 64D int8 eight quarters: 49.152 TB


In [3]:
train_patches = 62_000
hr_side = 1280 // 2
print(f'62k patch 2m 4-band uint16: {train_patches * hr_side**2 * 4 * 2 / 1e9:.3f} GB')
print(f'62k patch 32x32x1024 fp16: {train_patches * 32 * 32 * 1024 * 2 / 1e9:.3f} GB')
print(f'62k patch 128x128x64 int8: {train_patches * 128 * 128 * 64 / 1e9:.3f} GB')
print(f'62k patch 128x128x64 fp16: {train_patches * 128 * 128 * 64 * 2 / 1e9:.3f} GB')

62k patch 2m 4-band uint16: 203.162 GB
62k patch 32x32x1024 fp16: 130.023 GB
62k patch 128x128x64 int8: 65.012 GB
62k patch 128x128x64 fp16: 130.023 GB


## 解释

估算不含边界空瓦片、overlap、掩膜、索引、压缩差异、失败重跑、临时工作空间和副本。因此它们是容量下界，不是采购盘位。